### debugging

In [ ]:
import pandas as pd

CSV_PATH = '/home/louai/Ai-project-main/data/emplo_final.csv'
USE_COLS = ['sector', 'contract_type', 'edu_value', 'city', 'years_experience', 'salary']

# Define the hierarchy order (sector → contract → edu → city → exp → salary)
HIERARCHY = ['sector', 'contract_type', 'edu_value', 'city', 'years_experience']

def build_transition_model(df, current_level=0):
    if current_level >= len(HIERARCHY):
        return {}
    
    attr = HIERARCHY[current_level]
    transition = {}
    
    # Group by current attribute (e.g., sector)
    for value, group in df.groupby(attr, dropna=False):
        # Fill missing values for grouping
        value = str(value) if not pd.isna(value) else 'Unknown'
        
        # Compute expected values for remaining attributes
        expected = {}
        remaining_attrs = [a for a in HIERARCHY + ['salary'] if a not in HIERARCHY[:current_level+1]]
        
        for rem_attr in remaining_attrs:
            if rem_attr in ['years_experience', 'salary', 'edu_value']:
                # Numerical: mean
                expected[rem_attr] = round(group[rem_attr].mean(), 1)
            else:
                # Categorical: normalized counts
                counts = group[rem_attr].value_counts(dropna=False)
                total = counts.sum()
                expected[rem_attr] = {str(k): round(v/total, 2) for k, v in counts.items()}
        
        # Recursively build sub-clusters for next level
        sub_clusters = build_transition_model(group, current_level + 1)
        
        # Assemble node
        transition[value] = {
            'expected_values': expected,
            'clusters': sub_clusters
        }
    
    return transition

# Load data and build model
df = pd.read_csv(CSV_PATH, usecols=USE_COLS)
df['sector'] = df['sector'].fillna('Unknown').astype(str)
transition_model = build_transition_model(df)

def print_model_structure(model, indent=0, prefix="Cluster"):
    for key in model:
        # Print current cluster key (e.g., "IT", "Full-time")
        print(f"{' ' * indent}{prefix}: {key}")
        
        # Print expected values
        print(f"{' ' * (indent+2)}Expected Values:")
        for attr, value in model[key]['expected_values'].items():
            if isinstance(value, dict):
                formatted = ', '.join([f"{k}: {v:.2f}" if isinstance(v, float) else f"{k}: {v}" 
                                     for k, v in value.items()])
                print(f"{' ' * (indent+4)}{attr}: {{{formatted}}}")
            else:
                print(f"{' ' * (indent+4)}{attr}: {value:.1f}" if isinstance(value, float) 
                      else f"{' ' * (indent+4)}{attr}: {value}")
        
        # Recursively print sub-clusters
        if model[key]['clusters']:
            print(f"{' ' * (indent+2)}Sub-clusters:")
            print_model_structure(model[key]['clusters'], indent+4, "└──")

# Usage
print("=" * 50)
print("Transition Model Structure Validation")
print("=" * 50)
print_model_structure(transition_model)

In [40]:
import json

# Export the transition model to a JSON file
def export_transition_model(transition_model, file_path='transition_model.json'):
    """
    Exports the transition model as a JSON file.

    Args:
        transition_model (dict): The transition model to export.
        file_path (str): The file path to save the JSON file.
    """
    with open(file_path, 'w') as json_file:
        json.dump(transition_model, json_file, indent=4)
    print(f"Transition model exported to {file_path}")

# Import the transition model from a JSON file


# Example usage:
# Export the transition model
export_transition_model(transition_model, file_path='transition_model.json')

# Import the transition model in another notebook
# (Run this in the other notebook)
# imported_model = import_transition_model(file_path='transition_model.json')

Transition model exported to transition_model.json


In [ ]:

# use this function in the other files to import the json file

def import_transition_model(file_path='transition_model.json'):
    """
    Imports the transition model from a JSON file.

    Args:
        file_path (str): The file path of the JSON file to import.

    Returns:
        dict: The imported transition model.
    """
    with open(file_path, 'r') as json_file:
        transition_model = json.load(json_file)
    print(f"Transition model imported from {file_path}")
    return transition_model